# Lab 12 - Regression I


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

from scipy.interpolate import UnivariateSpline
from sklearn.metrics import mean_squared_error


In [ ]:
def g(x):
    return 4.26 * (np.exp(-x) - 4 * np.exp(-2 * x) + 3 * np.exp(-3 * x))


def make_regression(n=200, random_state=42):
    local_rng = np.random.default_rng(random_state)
    x = local_rng.uniform(0, 4, size=n)
    eps = local_rng.normal(0, 0.1, size=n)
    y = g(x) + eps
    return x, y


x_train, y_train = make_regression(200, RANDOM_STATE)
x_test, y_test = make_regression(1000, RANDOM_STATE + 1)
grid = np.linspace(0, 4, 400)


## Nadaraya-Watson from scratch and smoothing splines


In [ ]:
def gaussian_kernel(u):
    return np.exp(-0.5 * u ** 2) / np.sqrt(2 * np.pi)


def epanechnikov_kernel(u):
    return 0.75 * np.maximum(0, 1 - u ** 2)


def nadaraya_watson_predict(x_train, y_train, x_eval, h=0.2, kernel=gaussian_kernel):
    u = (x_eval[:, None] - x_train[None, :]) / h
    weights = kernel(u)
    denom = np.maximum(weights.sum(axis=1), 1e-12)
    return (weights @ y_train) / denom


def fit_spline_predict(x_train, y_train, x_eval, s=2.0):
    order = np.argsort(x_train)
    spline = UnivariateSpline(x_train[order], y_train[order], s=s)
    return spline(x_eval)


In [ ]:
bandwidths = [0.05, 0.15, 0.35, 0.7]
s_values = [0.1, 1.0, 5.0]

plt.figure(figsize=(12, 6))
plt.scatter(x_train, y_train, s=12, alpha=0.4, label="training data")
plt.plot(grid, g(grid), color="black", linewidth=3, label="true g(x)")
for h in bandwidths:
    plt.plot(grid, nadaraya_watson_predict(x_train, y_train, grid, h=h), label=f"NW h={h}")
for s in s_values:
    plt.plot(grid, fit_spline_predict(x_train, y_train, grid, s=s), linestyle="--", label=f"spline s={s}")
plt.legend(ncol=2)
plt.grid(True)
plt.show()


In [ ]:
def errors(name, pred_train, pred_test):
    return {
        "method": name,
        "train_MSE_g": mean_squared_error(g(x_train), pred_train),
        "test_MSE_g": mean_squared_error(g(x_test), pred_test),
        "train_MSE_y": mean_squared_error(y_train, pred_train),
        "test_MSE_y": mean_squared_error(y_test, pred_test),
    }


rows = []
for h in bandwidths:
    rows.append(errors(f"NW h={h}", nadaraya_watson_predict(x_train, y_train, x_train, h), nadaraya_watson_predict(x_train, y_train, x_test, h)))
for s in s_values:
    rows.append(errors(f"spline s={s}", fit_spline_predict(x_train, y_train, x_train, s), fit_spline_predict(x_train, y_train, x_test, s)))
rows


MSE_g compares the fitted function with the known true function and is available only in simulation. MSE_y compares predictions with observed responses and is available for real regression data.


In [ ]:
def sample_size_curve(ns=(50, 100, 200, 500, 1000), reps=20, h_values=(0.1, 0.25, 0.6)):
    curves = {h: [] for h in h_values}
    for n in ns:
        for h in h_values:
            vals = []
            for rep in range(reps):
                x_tr, y_tr = make_regression(n, 5000 + rep + n)
                x_te, y_te = make_regression(1000, 9000 + rep)
                pred = nadaraya_watson_predict(x_tr, y_tr, x_te, h=h)
                vals.append(mean_squared_error(y_te, pred))
            curves[h].append(np.mean(vals))
    return np.array(ns), curves


ns, curves = sample_size_curve(reps=10)
for h, vals in curves.items():
    plt.plot(ns, vals, marker="o", label=f"h={h}")
plt.xlabel("training sample size")
plt.ylabel("average test MSE_y")
plt.grid(True)
plt.legend()
plt.show()


Nadaraya-Watson is local: one observation mostly affects nearby predictions. Smoothing splines are more global because the fitted curve is controlled by a whole-function smoothness penalty. Smoothing selection is model selection; in real work use validation or cross-validation rather than training error, because training error favors overly rough curves.
